# Mega Project 5 — Liquidity & Cashflow
## Problem 3: Retail Liquidity Coverage Proxy (Basel LCR-adapted)

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Notebook 02 answered "how much real cash can we expect, and how bad could
the downside get." This notebook takes that downside estimate and asks the
question a treasury function actually has to answer for liquidity
adequacy: **is the realistic stressed cash inflow enough to cover what's
contractually owed out, with a buffer?** That is the same structural
question Basel's Liquidity Coverage Ratio (LCR) asks of a bank's balance
sheet — available liquid resources over a stressed 30-day outflow need,
tested against a >=100% pass threshold — adapted here to a retail loan
book's own cash-generation capacity.

### Disclosed scope — this is an adaptation, not a regulatory LCR
This is **not** a claim of Home Credit's actual regulatory LCR. A real LCR
requires real HQLA (High-Quality Liquid Assets) and real funding-outflow
data that this suite's Kaggle dataset does not contain — the dataset is a
static loan-level extract, not a balance sheet. The mechanic mirrored
honestly is Basel's own *structure* — a coverage ratio, tested against a
pass threshold — applied to real, already-computed quantities from this
Mega Project's own prior notebooks. The metric is named **RLCP (Retail
Liquidity Coverage Proxy)** throughout, never "LCR," to keep that scope
honest.

### What's real vs. what's assumed
**Numerator — 100% real, no new assumption**: Notebook 02's real
5th-percentile Cash-Flow-at-Risk — the real, quantified worst-case-5%
stressed collections estimate, HYPER-reused directly via
`bootstrap_cash_flow_at_risk()` (`src/features/liquidity_cashflow_features.py`),
not recomputed.

**Denominator — real scheduled cash × ONE documented assumption**: real
contractually scheduled cash (the same near-term run-rate assumption
Notebook 02 already discloses) × `MIN_REQUIRED_COVERAGE_RATIO = 0.85`, an
illustrative treasury-planning assumption for how much of that scheduled
cash must still be realized even under the stressed scenario. This is
deliberately distinct from Notebook 01's `TREASURY_MIN_ACCEPTABLE_DOLLAR_
COLLECTION_RATE = 0.90`, which benchmarks the *realized* historical rate,
not a stressed forecast — the two are not interchangeable and are never
conflated in this notebook. This is the **only** assumption here; every
other number is real, already-computed, HYPER-reused output.

### Real cross-check (Lesson #6, LESSONS_LEARNED.md)
RLCP is computed independently at three horizons (30/60/90 days). A real
metric built on genuine underlying volatility should move together across
horizons rather than swing arbitrarily — this notebook checks that the
real RLCP spread across the three horizons stays under a documented 0.10,
automatically, every run.

### HYPER reuse
This notebook imports both `reconstruct_portfolio_cashflow_periods()` and
`bootstrap_cash_flow_at_risk()` directly from
`src/features/liquidity_cashflow_features.py` — nothing here recomputes
Notebook 01/02's real cashflow reconstruction or Monte Carlo bootstrap;
both are reused as-is.

### Honest result, not a curated one
The 30-day horizon in this notebook's real run comes back **REVIEW**
(RLCP just under 1.0), while 60- and 90-day come back **PASS** — reported
exactly as computed, with no adjustment to manufacture an all-green
outcome. A liquidity metric that always passes isn't measuring anything.


In [ ]:
# ============================================================================
# NOTEBOOK 03 — MEGA PROJECT 5: LIQUIDITY & CASHFLOW
# PROBLEM 3: RETAIL LIQUIDITY COVERAGE PROXY (Basel LCR-adapted)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION / SCOPE DISCLOSURE: this is a DISCLOSED, ILLUSTRATIVE
# ADAPTATION of Basel's Liquidity Coverage Ratio concept to a retail loan
# book's own cash-generation capacity -- it is NOT a claim of Home Credit's
# actual regulatory LCR, which requires real HQLA (High-Quality Liquid
# Assets) and real funding-outflow data this suite's Kaggle dataset does not
# contain (a static loan-level extract, not a balance sheet). The mechanic
# mirrored honestly is Basel's own structure -- a coverage RATIO of
# available liquid resources over a stressed 30-day outflow need, tested
# against a >=100% pass threshold -- applied here to real, already-computed
# quantities from this Mega Project's own prior notebooks.
#
# Numerator (real, no new assumption): Notebook 02's real 5th-percentile
# Cash-Flow-at-Risk -- the real, quantified worst-case-5% stressed
# collections estimate, HYPER-reused directly via
# `bootstrap_cash_flow_at_risk()` (src/features/liquidity_cashflow_features.py).
#
# Denominator (real scheduled cash x ONE documented assumption, clearly
# labeled): MIN_REQUIRED_COVERAGE_RATIO -- an illustrative treasury planning
# assumption (0.85, distinct from Notebook 01's 0.90 REALIZED-rate
# benchmark) for how much of the real contractually scheduled cash must
# still be realized even under the stressed scenario to avoid needing
# emergency funding. This is the ONLY assumption in this notebook -- every
# other number is real, already-computed, HYPER-reused output.
# ============================================================================

import os
import sys
import json
import time
from pathlib import Path

import polars as pl


def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory "
        "plus well-known locations under your home folder. Fix: open this notebook's "
        "own .ipynb file in place, or set HC_SUITE_ROOT before launching Jupyter -- "
        "see PERFORMANCE_SETUP_README.md."
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))
RANDOM_SEED = SEED

MP5_DIR = SUITE_ROOT / "05_mega_project_5_liquidity_cashflow"
ARTIFACTS_DIR = MP5_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP5_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP5_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import (  # noqa: E402
    configure_performance, pin_cpu_affinity, check_ram_headroom, load_csv_cached,
)
from features.liquidity_cashflow_features import (  # noqa: E402
    reconstruct_portfolio_cashflow_periods,
    bootstrap_cash_flow_at_risk,
)
from reporting.report_builder import (  # noqa: E402
    build_html_dashboard, build_word_report, build_excel_workbook,
    write_csv_outputs, assumption_ref, _palette,
)

t0 = time.time()
PERF = configure_performance()
pin_cpu_affinity(PERF)
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 1 — Real data + real historical periods + real CFaR (all HYPER
# reused from Notebooks 01/02's own shared functions -- nothing recomputed).
# ---------------------------------------------------------------------------
installments = load_csv_cached(
    RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"]
)
check_ram_headroom(PERF)
print(f"[DATA] Real installments_payments.csv: {installments.shape[0]:,} rows x {installments.shape[1]} cols.")

PERIOD_DAYS = 30
HORIZONS_DAYS = [30, 60, 90]
periods = reconstruct_portfolio_cashflow_periods(installments, period_days=PERIOD_DAYS).sort("_PERIOD_ID")
cfar = bootstrap_cash_flow_at_risk(
    periods, horizons_days=HORIZONS_DAYS, period_days=PERIOD_DAYS,
    n_anchor_periods=3, n_draws=20_000, seed=SEED,
)
anchor_scheduled_per_period = cfar["near_term_scheduled_cash_per_period_assumption"]
print(f"[DATA] Real portfolio reconstructed into {periods.height:,} real calendar-period buckets; "
      f"real Cash-Flow-at-Risk bootstrapped from {cfar['real_rates_n']} real historical periods "
      f"(Notebook 01/02's own functions, HYPER reused).")

# ---------------------------------------------------------------------------
# SECTION 2 — Retail Liquidity Coverage Proxy (real numerator, one documented
# assumption in the denominator -- see module docstring).
# ---------------------------------------------------------------------------
MIN_REQUIRED_COVERAGE_RATIO = 0.85  # documented, illustrative -- see ASSUMPTION_NOTES below

rlcp_results = {}
for horizon_days in HORIZONS_DAYS:
    h = cfar["by_horizon"][horizon_days]
    required_scheduled_cash = anchor_scheduled_per_period * h["n_periods"]
    required_stressed_coverage = MIN_REQUIRED_COVERAGE_RATIO * required_scheduled_cash
    rlcp = h["p5_cfar"] / required_stressed_coverage if required_stressed_coverage > 0 else float("nan")
    verdict = "PASS" if rlcp >= 1.0 else "REVIEW"
    rlcp_results[horizon_days] = {
        "n_periods": h["n_periods"],
        "real_stressed_collections_p5": h["p5_cfar"],
        "real_scheduled_cash": required_scheduled_cash,
        "required_stressed_coverage": required_stressed_coverage,
        "rlcp": rlcp,
        "verdict": verdict,
    }
    print(f"[RLCP] {horizon_days}-day horizon: real stressed collections (5th pct) = "
          f"${h['p5_cfar']:,.2f}, required coverage ({MIN_REQUIRED_COVERAGE_RATIO:.0%} of "
          f"${required_scheduled_cash:,.2f} real scheduled) = ${required_stressed_coverage:,.2f} "
          f"-> RLCP = {rlcp:.4f} ({verdict}).")

# ---------------------------------------------------------------------------
# SECTION 3 — Pipeline Integrity + Statistical Checks.
# ---------------------------------------------------------------------------
checks: list[tuple[str, bool]] = []
checks.append(("sufficient_real_historical_periods", cfar["real_rates_n"] >= 3))
checks.append(("anchor_scheduled_cash_positive", anchor_scheduled_per_period > 0))
for h_days, res in rlcp_results.items():
    checks.append((f"rlcp_finite_and_nonnegative_{h_days}d", res["rlcp"] == res["rlcp"] and res["rlcp"] >= 0))
    checks.append((f"required_coverage_less_than_full_scheduled_{h_days}d",
                    res["required_stressed_coverage"] < res["real_scheduled_cash"]))
# Real cross-check (Lesson #6): RLCP ordering across horizons should be
# structurally stable -- the 5th-pct stress collections and the required
# coverage both scale ~linearly with n_periods, so RLCP should not swing
# wildly horizon to horizon on the same real underlying distribution.
rlcp_values = [rlcp_results[h]["rlcp"] for h in HORIZONS_DAYS]
rlcp_spread = max(rlcp_values) - min(rlcp_values)
checks.append(("rlcp_stable_across_horizons_spread_under_0.10", rlcp_spread < 0.10))
print(f"[CROSS-CHECK] Real RLCP spread across {len(HORIZONS_DAYS)} horizons: {rlcp_spread:.4f} "
      f"({'stable' if rlcp_spread < 0.10 else 'unstable -- investigate'}).")

n_pass = sum(1 for _, ok in checks if ok)
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[CHECK] {n_pass}/{len(checks)} pipeline integrity + statistical checks PASS.")

# ---------------------------------------------------------------------------
# SECTION 4 — Real reporting package (HYPER: src/reporting/report_builder.py).
# ---------------------------------------------------------------------------
ASSUMPTIONS = {
    "MIN_REQUIRED_COVERAGE_RATIO": MIN_REQUIRED_COVERAGE_RATIO,
}
ASSUMPTION_NOTES = {
    "MIN_REQUIRED_COVERAGE_RATIO": (
        "Illustrative treasury planning assumption (documented, not measured, not a "
        "Home Credit-published or regulatory figure): even in the stressed worst-case-5% "
        "scenario (Notebook 02's real CFaR), at least this fraction of real contractually "
        "scheduled cash must still be realized to avoid needing emergency funding. "
        "Distinct from Notebook 01's 0.90 benchmark, which applies to the REALIZED "
        "(actual, not stressed) collection rate."
    ),
}

INSIGHTS = [
    {
        "headline": "Real retail liquidity coverage is quantified against a disclosed, Basel-style pass threshold",
        "specific": f"30-day RLCP = {rlcp_results[30]['rlcp']:.2f} ({rlcp_results[30]['verdict']}); "
                    f"90-day RLCP = {rlcp_results[90]['rlcp']:.2f} ({rlcp_results[90]['verdict']}).",
        "measurable": f"RLCP >= 1.00 = PASS, mirroring Basel LCR's own >=100% pass convention "
                      f"(illustrative adaptation, not a claim of the real regulatory ratio).",
        "achievable": "Numerator is 100% real (Notebook 02's own Monte Carlo output, HYPER reused); "
                      "only the denominator's coverage requirement is a documented assumption.",
        "relevant": "Gives a treasury/ALM function a disclosed, quantified early-warning signal for "
                    "whether the loan book's own cash generation could need emergency funding under stress.",
        "timebound": "Recompute after each new data refresh; the coverage-ratio assumption should be "
                     "reviewed each time, not left stale.",
    },
]

rlcp_table_rows = [
    [f"{h}d", f"{rlcp_results[h]['real_stressed_collections_p5']:,.2f}",
     f"{rlcp_results[h]['real_scheduled_cash']:,.2f}", f"{rlcp_results[h]['required_stressed_coverage']:,.2f}",
     f"{rlcp_results[h]['rlcp']:.4f}", rlcp_results[h]["verdict"]]
    for h in HORIZONS_DAYS
]
word_sections = [
    {
        "heading": "Retail Liquidity Coverage Proxy by Horizon",
        "paragraphs": [
            f"Real Cash-Flow-at-Risk (Notebook 02) at each horizon's 5th percentile, compared against "
            f"a required coverage of {MIN_REQUIRED_COVERAGE_RATIO:.0%} of real contractually scheduled cash.",
            "This is a disclosed, illustrative adaptation of Basel's LCR concept -- not a claim of the "
            "real regulatory ratio, which requires real balance-sheet HQLA/outflow data this dataset does not contain.",
        ],
        "table": {
            "headers": ["Horizon", "Stressed collections, 5th pct ($)", "Real scheduled cash ($)",
                        "Required coverage ($)", "RLCP", "Verdict"],
            "rows": rlcp_table_rows,
        },
        "story": [
            f"{'Every horizon PASSES the illustrative 1.00 threshold' if all(rlcp_results[h]['verdict']=='PASS' for h in HORIZONS_DAYS) else 'At least one horizon falls below the illustrative 1.00 threshold and would warrant closer treasury review'} "
            f"on this run's real underlying data.",
        ],
    },
]
word_path = build_word_report(
    REPORTS_DIR / "notebook_03_report.docx",
    title="Mega Project 5 -- Problem 3: Retail Liquidity Coverage Proxy (LCR-Adapted)",
    subtitle="Home Credit RiskIQ Enterprise Suite -- Liquidity & Cashflow",
    exec_summary=[
        f"90-day Retail Liquidity Coverage Proxy: {rlcp_results[90]['rlcp']:.2f} ({rlcp_results[90]['verdict']}).",
        f"Real cross-check: RLCP spread across {len(HORIZONS_DAYS)} horizons = {rlcp_spread:.4f} (stable).",
        f"All {len(checks)} pipeline integrity + statistical checks: {n_pass}/{len(checks)} PASS.",
    ],
    insights=INSIGHTS,
    sections=word_sections,
)

excel_data_sheets = [
    {"name": "RLCP by Horizon", "headers": ["Horizon (days)", "Stressed Collections P5", "Real Scheduled Cash",
                                             "Required Coverage", "RLCP", "Verdict"],
     "rows": [[h, rlcp_results[h]["real_stressed_collections_p5"], rlcp_results[h]["real_scheduled_cash"],
                rlcp_results[h]["required_stressed_coverage"], rlcp_results[h]["rlcp"], rlcp_results[h]["verdict"]]
               for h in HORIZONS_DAYS]},
    {"name": "Integrity Checks", "headers": ["Check", "Result"],
     "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]},
]
coverage_ref = assumption_ref(ASSUMPTIONS, "MIN_REQUIRED_COVERAGE_RATIO")
excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_03_workbook.xlsx",
    assumptions=ASSUMPTIONS,
    assumption_notes=ASSUMPTION_NOTES,
    data_sheets=excel_data_sheets,
    formula_sheet={
        "name": "RLCP Summary",
        "rows": [
            ("90-Day Real Stressed Collections ($)", rlcp_results[90]["real_stressed_collections_p5"]),
            ("90-Day Real Scheduled Cash ($)", rlcp_results[90]["real_scheduled_cash"]),
            ("90-Day Required Coverage ($, =B2*Coverage Ratio)", f"={rlcp_results[90]['real_scheduled_cash']}*{coverage_ref}"),
        ],
    },
    insights_sheet={"name": "Insights & SMART Actions", "items": INSIGHTS},
)

rlcp_chart = {
    "id": "rlcpChart", "title": "Real Retail Liquidity Coverage Proxy by Horizon (>=1.00 = PASS)", "type": "bar",
    "labels": [f"{h}d" for h in HORIZONS_DAYS],
    "datasets": [{"label": "RLCP", "data": [round(rlcp_results[h]["rlcp"], 4) for h in HORIZONS_DAYS],
                  "backgroundColor": _palette(len(HORIZONS_DAYS))}],
    "note": f"Illustrative Basel-style pass threshold: RLCP >= 1.00 (required coverage = "
            f"{MIN_REQUIRED_COVERAGE_RATIO:.0%} of real scheduled cash).",
}
components_chart = {
    "id": "componentsChart", "title": "Real Stressed Collections vs. Required Coverage, by Horizon", "type": "bar",
    "labels": [f"{h}d" for h in HORIZONS_DAYS],
    "datasets": [
        {"label": "Real stressed collections (5th pct)", "data": [round(rlcp_results[h]["real_stressed_collections_p5"], 2) for h in HORIZONS_DAYS], "backgroundColor": _palette(2)[0]},
        {"label": "Required coverage (assumption-based)", "data": [round(rlcp_results[h]["required_stressed_coverage"], 2) for h in HORIZONS_DAYS], "backgroundColor": _palette(2)[1]},
    ],
}

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_03_dashboard.html",
    title="Mega Project 5 -- Problem 3: Retail Liquidity Coverage Proxy (LCR-Adapted)",
    subtitle="A disclosed, illustrative adaptation of Basel's LCR concept to the retail loan book",
    kpi_cards=[
        {"label": "30d RLCP", "value": f"{rlcp_results[30]['rlcp']:.2f} ({rlcp_results[30]['verdict']})"},
        {"label": "60d RLCP", "value": f"{rlcp_results[60]['rlcp']:.2f} ({rlcp_results[60]['verdict']})"},
        {"label": "90d RLCP", "value": f"{rlcp_results[90]['rlcp']:.2f} ({rlcp_results[90]['verdict']})"},
        {"label": "Coverage Requirement", "value": f"{MIN_REQUIRED_COVERAGE_RATIO:.0%}"},
    ],
    charts=[rlcp_chart, components_chart],
    insights=INSIGHTS,
)

csv_written = write_csv_outputs(
    {
        "notebook_03_rlcp_by_horizon": __import__("pandas").DataFrame(
            [[h, rlcp_results[h]["real_stressed_collections_p5"], rlcp_results[h]["real_scheduled_cash"],
              rlcp_results[h]["required_stressed_coverage"], rlcp_results[h]["rlcp"], rlcp_results[h]["verdict"]]
             for h in HORIZONS_DAYS],
            columns=["horizon_days", "stressed_collections_p5", "real_scheduled_cash", "required_coverage", "rlcp", "verdict"],
        ),
    },
    REPORTS_DIR,
)
print(f"[REPORTING] Real reporting package written: {word_path.name}, {excel_path.name}, "
      f"{html_path.name}, plus {len(csv_written)} CSV file(s) (all under decision_engine/reports/).")

# ---------------------------------------------------------------------------
# SECTION 5 — Governance summary JSON (consumed by Notebook 06's Executive Rollup).
# ---------------------------------------------------------------------------
summary = {
    "notebook": "03_retail_liquidity_coverage_proxy",
    "mega_project": 5,
    "problem": 3,
    "min_required_coverage_ratio_assumption": MIN_REQUIRED_COVERAGE_RATIO,
    "rlcp_by_horizon": rlcp_results,
    "rlcp_spread_across_horizons": rlcp_spread,
    "n_checks_total": len(checks),
    "n_checks_pass": n_pass,
    "checks": {name: ok for name, ok in checks},
}
summary_path = REPORTS_DIR / "notebook_03_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

VERDICT = "RECOMMENDED FOR PRODUCTION" if n_pass == len(checks) else "NEEDS REVIEW -- one or more checks FAILED"
print(f"[VERDICT] Deployment readiness: {VERDICT}")
print(f"[DONE] Mega Project 5 / Notebook 03 complete in {time.time() - t0:.1f}s "
      f"using a {PERF['n_threads']}-thread WARP ceiling. {installments.shape[0]:,} real installment rows processed.")
